# Cloud Provider Analytics — Full Pipeline

```text
Landing → Bronze → Silver → Gold → Serving (AstraDB)
```

End-to-end orchestration. Logic lives in `src/jobs/`; this notebook runs each layer.

**Prerequisites:** `pipeline.ipynb` at repo root (cwd = `cloud-provider-analytics/`), `datalake/landing/`, Astra keyspace `cloud_analytics`, token + bundle.

In [4]:
# Setup local + Spark
import os
import sys
from pathlib import Path

# Repo root: notebook cwd (pipeline.ipynb at project root)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(
        "Open Jupyter/VS Code with cwd = repo root (cloud-provider-analytics/)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.spark.java import ensure_java_home

JAVA_HOME = ensure_java_home()

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

from src.cassandra.client import is_astra_configured
from src.config import BRONZE, CASSANDRA_KEYSPACE, DATA_ROOT, GOLD, LANDING, SILVER
from src.jobs.bronze_streaming import (
    USAGE_EVENTS_BRONZE_PATH,
    USAGE_EVENTS_CHECKPOINT_PATH,
    USAGE_EVENTS_LANDING_GLOB,
)
from src.jobs.gold import ORG_DAILY_USAGE_BY_SERVICE
from src.jobs.silver import USAGE_EVENTS_SILVER
from src.schemas.bronze_streaming import WATERMARK_DELAY
from src.spark.performance import configure_spark_performance

spark = (
    SparkSession.builder.appName("cloud-provider-analytics")
    .master("local[*]")
    .getOrCreate()
)
configure_spark_performance(spark)
spark.sparkContext.setLogLevel("WARN")

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"DATA_ROOT:     {DATA_ROOT}")
print(f"JAVA_HOME:     {JAVA_HOME}")
print(f"Spark:         {spark.version}")
print(f"Astra:         {is_astra_configured()}")
print(f"KEYSPACE:      {CASSANDRA_KEYSPACE}")




Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/11 02:33:33 WARN Utils: Your hostname, cnt-desktop, resolves to a loopback address: 127.0.1.1; using 192.168.0.141 instead (on interface wlx5ca6e63a03f8)
26/07/11 02:33:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/11 02:33:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PROJECT_ROOT:  /home/cnt/Desktop/tp-big-data
DATA_ROOT:     /home/cnt/Desktop/tp-big-data/datalake
JAVA_HOME:     /usr/lib/jvm/java-21-openjdk-amd64
Spark:         4.1.2
Astra:         False
KEYSPACE:      cloud_analytics


## 1. Batch Bronze — master CSVs

Ingest 7 landing masters → Parquet with `ingest_ts`, `source_file`, and dedupe.

In [5]:
from src.jobs.bronze_batch import run_batch_bronze, validate_bronze_uniqueness

batch_results = run_batch_bronze(spark)
display(pd.DataFrame(batch_results)[
    ["dataset_name", "raw_count", "deduped_count", "removed_duplicates", "written_count"]
])
display(pd.DataFrame(validate_bronze_uniqueness(spark)))


,dataset_name,raw_count,deduped_count,removed_duplicates,written_count
0,customers_orgs,80,80,0,80
1,users,800,800,0,800
2,billing_monthly,240,240,0,240
3,resources,400,400,0,400
4,support_tickets,1000,1000,0,1000
5,marketing_touches,1500,1500,0,1500
6,nps_surveys,92,92,0,92


,dataset_name,dedup_keys,total_rows,distinct_keys,is_unique
0,customers_orgs,[org_id],80,80,True
1,users,[user_id],800,800,True
2,billing_monthly,[invoice_id],240,240,True
3,resources,[resource_id],400,400,True
4,support_tickets,[ticket_id],1000,1000,True
5,marketing_touches,[touch_id],1500,1500,True
6,nps_surveys,"[org_id, survey_date]",92,92,True


## 2. Streaming Bronze — usage_events

Structured Streaming from JSONL, watermark, dedupe `event_id`, late data, and reparquet.

> **Clean run:** `reset_state=True` rebuilds Bronze from landing with early normalization.
> Replay: `ingest_ts = event_ts` (watermark 60d). Production: `ingest_ts = now()`.

In [6]:
from src.jobs.bronze_streaming import run_streaming_bronze, validate_bronze_streaming

print(f"Landing: {USAGE_EVENTS_LANDING_GLOB}")
streaming_result = run_streaming_bronze(spark, reset_state=True)
display(pd.DataFrame([{k: v for k, v in streaming_result.items() if k != "reparquet"}]))
display(pd.DataFrame([validate_bronze_streaming(spark)]))


Landing: /home/cnt/Desktop/tp-big-data/datalake/landing/usage_events_stream/*.jsonl


26/07/11 02:33:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/11 02:33:55 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/07/11 02:33:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/11 02:33:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/11 02:33:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/11 02:33:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/07/11 02:33:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group s

,landing_glob,bronze_path,checkpoint_path,watermark_delay,watermark_delay_production,late_data_threshold_sec,late_catchup_max_sec,ingest_ts_mode,written_count,distinct_event_ids,is_unique,late_arrivals,schema_v2_rows,streaming_input_rows
0,/home/cnt/Desktop/tp-big-data/datalake/landing...,/home/cnt/Desktop/tp-big-data/datalake/bronze/...,/home/cnt/Desktop/tp-big-data/datalake/checkpo...,60 days,10 minutes,600,1800,event_ts,43200,43200,True,0,32400,43200


,total_rows,distinct_event_ids,is_unique,missing_columns,columns_ok
0,43200,43200,True,[],True


## 3. Silver

Masters + `usage_events` (event grain), `org_service_daily`, cost anomalies, and quarantine.

In [17]:
from src.jobs.silver import run_silver, validate_silver

silver_results = run_silver(spark)

masters = [r for r in silver_results if r["dataset_name"] != "usage_events"]
events = next(r for r in silver_results if r["dataset_name"] == "usage_events")

print("Silver — maestros")
display(
    pd.DataFrame(masters)[
        ["dataset_name", "raw_count", "silver_count", "quarantine_count", "balance_ok"]
    ]
)

print("Silver — usage_events")
display(
    pd.DataFrame(
        [
            {
                "bronze": events["raw_count"],
                "silver_valid": events["valid_count"],
                "quarantine": events["quarantine_count"],
                "org_service_daily": events["org_service_daily_count"],
                "balance_ok": events["balance_ok"],
                "cost_anomalies": events["cost_anomalies_flagged"],
                "late_flagged": events["late_arrivals_flagged"],
                "late_quarantined": events["late_arrivals_quarantined"],
                "negative_cost_quarantined": events["negative_cost_quarantined"],
            }
        ]
    )
)

if events.get("quarantine_sample"):
    print("Quarantine sample (usage_events)")
    display(pd.DataFrame(events["quarantine_sample"]))

validation = validate_silver(spark)
print("Validación Silver — maestros")
display(pd.DataFrame(validation["masters"]))
print("Validación Silver — usage_events")
display(pd.DataFrame([validation["usage_events"]]))
print("Validación Silver — org_service_daily")
display(pd.DataFrame([validation["org_service_daily"]]))



26/07/11 03:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/11 03:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/11 03:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/11 03:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/11 03:05:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Silver — maestros


,dataset_name,raw_count,silver_count,quarantine_count,balance_ok
0,customers_orgs,80,80,0,True
1,users,800,800,0,True
2,billing_monthly,240,240,0,True
3,resources,400,400,0,True
4,support_tickets,1000,1000,0,True
5,marketing_touches,1500,1500,0,True
6,nps_surveys,92,92,0,True


Silver — usage_events


,bronze,silver_valid,quarantine,org_service_daily,balance_ok,cost_anomalies,late_flagged,late_quarantined,negative_cost_quarantined
0,43200,40956,2244,12108,True,10507,0,0,206


...

,dataset_name,bronze_count,silver_count,quarantine_count,balance_ok
0,customers_orgs,80,80,0,True
1,users,800,800,0,True
2,billing_monthly,240,240,0,True
3,resources,400,400,0,True
4,support_tickets,1000,1000,0,True
5,marketing_touches,1500,1500,0,True
6,nps_surveys,92,92,0,True


Validación Silver — usage_events


,bronze_count,silver_valid_count,quarantine_count,balance_ok,event_id_unique,missing_features,features_ok
0,43200,40956,2244,True,True,[],True


Validación Silver — org_service_daily


,silver_row_count,distinct_grain,grain_unique,missing_features,features_ok,cost_balance_ok
0,12108,12108,True,[],True,True


## 4. Gold — business marts

**Astra (5 tables):** daily FinOps, top services (rolling 14d), tickets, revenue, GenAI.

**Parquet only:** `cost_anomaly_mart`, `nps_by_org_date`, `marketing_touches_by_org_channel`.

In [8]:
from src.jobs.gold import run_gold, validate_gold

display(pd.DataFrame(run_gold(spark)))
display(pd.DataFrame([validate_gold(spark)]))


,mart_name,gold_path,gold_row_count,written_count,distinct_grain,grain_unique,silver_daily_row_count,silver_total_cost_usd,gold_total_cost_usd,cost_balance_ok,...,ticket_balance_ok,silver_genai_tokens,gold_genai_tokens,token_balance_ok,silver_survey_count,gold_survey_count,survey_balance_ok,silver_touch_count,gold_touch_count,touch_balance_ok
0,org_daily_usage_by_service,/home/cnt/Desktop/tp-big-data/datalake/gold/or...,12108,12108,12108,True,12108.0,141241.4284,141241.4284,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,org_top_services_by_cost,/home/cnt/Desktop/tp-big-data/datalake/gold/or...,258,258,258,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,revenue_by_org_month,/home/cnt/Desktop/tp-big-data/datalake/gold/re...,240,240,240,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cost_anomaly_mart,/home/cnt/Desktop/tp-big-data/datalake/gold/co...,12108,12108,12108,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,tickets_by_org_date,/home/cnt/Desktop/tp-big-data/datalake/gold/ti...,984,984,984,True,NaN,NaN,NaN,NaN,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,genai_tokens_by_org_date,/home/cnt/Desktop/tp-big-data/datalake/gold/ge...,1235,1235,1235,True,NaN,NaN,NaN,NaN,...,NaN,2418410.0,2418410.0,True,NaN,NaN,NaN,NaN,NaN,NaN
6,nps_by_org_date,/home/cnt/Desktop/tp-big-data/datalake/gold/np...,92,92,92,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,92.0,92.0,True,NaN,NaN,NaN
7,marketing_touches_by_org_channel,/home/cnt/Desktop/tp-big-data/datalake/gold/ma...,1477,1477,1477,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,True


,org_daily_usage_by_service,org_top_services_by_cost,revenue_by_org_month,cost_anomaly_mart,tickets_by_org_date,genai_tokens_by_org_date,nps_by_org_date,marketing_touches_by_org_channel
0,"{'gold_row_count': 12108, 'distinct_grain': 12...","{'gold_row_count': 258, 'distinct_grain': 258,...","{'gold_row_count': 240, 'distinct_grain': 240,...","{'gold_row_count': 12108, 'distinct_grain': 12...","{'gold_row_count': 984, 'distinct_grain': 984,...","{'gold_row_count': 1235, 'distinct_grain': 123...","{'gold_row_count': 92, 'distinct_grain': 92, '...","{'gold_row_count': 1477, 'distinct_grain': 147..."


## 5. Serving — load Gold → Astra

Load via `foreachBatch`. CQL queries in section 7.

In [9]:
from src.jobs.serving_cassandra import run_serving

if not is_astra_configured():
    print("Skipped: set ASTRA_DB_APPLICATION_TOKEN and ASTRA_DB_SECURE_BUNDLE_PATH.")
else:
    load = run_serving(spark).get("load")
    if load:
        display(pd.DataFrame(load))


Skipped: set ASTRA_DB_APPLICATION_TOKEN and ASTRA_DB_SECURE_BUNDLE_PATH.


## 6. Idempotency

Re-run batch Bronze → streaming (no `reset_state`) → Silver → Gold.

**Criterio de idempotencia:** conteos `before == after` en todas las capas, sin filas nuevas en streaming (`streaming_input_rows = 0`) y reparquet omitido. La unicidad de `event_id` en Bronze se reporta aparte; el balance Silver (válidos + cuarentena) puede diferir si un evento cae en más de una regla de cuarentena.

In [10]:
from src.jobs.bronze_batch import run_batch_bronze
from src.jobs.bronze_streaming import read_bronze_usage_events_parquet, run_streaming_bronze
from src.jobs.gold import (
    GENAI_TOKENS_BY_ORG_DATE,
    ORG_DAILY_USAGE_BY_SERVICE,
    REVENUE_BY_ORG_MONTH,
    TICKETS_BY_ORG_DATE,
    run_gold,
)
from src.jobs.silver import USAGE_EVENTS_QUARANTINE, run_silver

MASTER_DATASETS = [
    "customers_orgs", "users", "billing_monthly", "resources",
    "support_tickets", "marketing_touches", "nps_surveys",
]


def lake_snapshot() -> dict[str, int]:
    bronze_events = read_bronze_usage_events_parquet(spark)
    silver_valid = spark.read.parquet(USAGE_EVENTS_SILVER)
    quarantine = 0
    if os.path.isdir(USAGE_EVENTS_QUARANTINE):
        quarantine = spark.read.parquet(USAGE_EVENTS_QUARANTINE).count()
    return {
        "bronze_masters": sum(spark.read.parquet(f"{BRONZE}/{d}").count() for d in MASTER_DATASETS),
        "bronze_events": bronze_events.count(),
        "bronze_events_distinct": bronze_events.select("event_id").distinct().count(),
        "silver_valid": silver_valid.count(),
        "silver_quarantine": quarantine,
        "gold_finops": spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE).count(),
        "gold_revenue": spark.read.parquet(REVENUE_BY_ORG_MONTH).count(),
        "gold_tickets": spark.read.parquet(TICKETS_BY_ORG_DATE).count(),
        "gold_genai": spark.read.parquet(GENAI_TOKENS_BY_ORG_DATE).count(),
    }


before = lake_snapshot()
run_batch_bronze(spark)
streaming_rerun = run_streaming_bronze(spark, reset_state=False)
run_silver(spark)
run_gold(spark)
after = lake_snapshot()

comparison = pd.DataFrame(
    [{"metric": k, "before": before[k], "after": after[k], "ok": before[k] == after[k]}
     for k in before]
)
display(comparison)

reparquet_skipped = streaming_rerun["reparquet"].get("skipped", False)
silver_balance = after["silver_valid"] + after["silver_quarantine"]
silver_balance_ok = after["bronze_events"] == silver_balance

idempotency_checks = pd.DataFrame([
    {
        "check": "lake_counts_stable",
        "detail": "before == after en todas las métricas",
        "ok": comparison["ok"].all(),
    },
    {
        "check": "streaming_no_new_input",
        "detail": f"streaming_input_rows={streaming_rerun['streaming_input_rows']}",
        "ok": streaming_rerun["streaming_input_rows"] == 0,
    },
    {
        "check": "streaming_reparquet_skipped",
        "detail": str(streaming_rerun["reparquet"].get("reason", "n/a")),
        "ok": reparquet_skipped,
    },
    {
        "check": "bronze_event_id_unique",
        "detail": (
            f"rows={after['bronze_events']} distinct={after['bronze_events_distinct']}"
        ),
        "ok": after["bronze_events"] == after["bronze_events_distinct"],
    },
    {
        "check": "silver_row_balance",
        "detail": (
            f"bronze={after['bronze_events']} "
            f"valid+quarantine={silver_balance}"
        ),
        "ok": silver_balance_ok,
    },
])
display(idempotency_checks)

lake_ok = idempotency_checks["ok"].all()
print(
    f"Lake idempotency: {'OK' if lake_ok else 'FAIL'} | "
    f"bronze_total={streaming_rerun['written_count']} "
    f"new_stream_rows={streaming_rerun['streaming_input_rows']} "
    f"reparquet_skipped={reparquet_skipped}"
)

if is_astra_configured():
    from src.cassandra.client import get_cassandra_session
    from src.cassandra.schema import TABLE_ORG_DAILY
    from src.jobs.serving_cassandra import load_org_daily_usage_by_service

    sample_org = (
        spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
        .select("org_id")
        .limit(1)
        .collect()[0]["org_id"]
    )
    session, cluster = get_cassandra_session()
    try:
        before_c = session.execute(
            f"SELECT COUNT(*) FROM {TABLE_ORG_DAILY} WHERE org_id = %s",
            (sample_org,),
        ).one()[0]
        load_org_daily_usage_by_service(spark, session)
        after_c = session.execute(
            f"SELECT COUNT(*) FROM {TABLE_ORG_DAILY} WHERE org_id = %s",
            (sample_org,),
        ).one()[0]
        cassandra_ok = before_c == after_c
        display(pd.DataFrame([{
            "sample_org_id": sample_org,
            "rows_before": before_c,
            "rows_after": after_c,
            "ok": cassandra_ok,
        }]))
        print(f"Cassandra idempotency (sample org): {'OK' if cassandra_ok else 'FAIL'}")
    finally:
        cluster.shutdown()
else:
    print("Cassandra idempotency: skipped (Astra not configured).")


26/07/11 02:36:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/11 02:36:43 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/07/11 02:37:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


,metric,before,after,ok
0,bronze_masters,4112,4112,True
1,bronze_events,43200,43200,True
2,bronze_events_distinct,43200,43200,True
3,silver_valid,40956,40956,True
4,silver_quarantine,2244,2244,True
5,gold_finops,12108,12108,True
6,gold_revenue,240,240,True
7,gold_tickets,984,984,True
8,gold_genai,1235,1235,True


,check,detail,ok
0,lake_counts_stable,before == after en todas las métricas,True
1,streaming_no_new_input,streaming_input_rows=0,True
2,streaming_reparquet_skipped,no_new_stream_rows,True
3,bronze_event_id_unique,rows=43200 distinct=43200,True
4,silver_row_balance,bronze=43200 valid+quarantine=43200,True


Lake idempotency: OK | bronze_total=43200 new_stream_rows=0 reparquet_skipped=True
Cassandra idempotency: skipped (Astra not configured).


## 7. AstraDB queries (#1–#5)

One cell per query: **literal CQL** printed and executed (`src.cassandra.selects`, aligned with `cql/01–05`).

Prerequisite: section 5 or tables already loaded in `cloud_analytics`.

**Capturas de referencia** (corrida con Astra configurado): ver `documentation/query_capture_examples/`

| Query | Captura |
|---|---|
| #1 Costos/requests diarios | ![Q1](documentation/query_capture_examples/query1.png) |
| #2 Top-N servicios (14d) | ![Q2](documentation/query_capture_examples/query2.png) |
| #3 Tickets críticos + SLA | ![Q3](documentation/query_capture_examples/query3.png) |
| #4 Revenue mensual USD | ![Q4](documentation/query_capture_examples/query4.png) |
| #5 Tokens GenAI | ![Q5](documentation/query_capture_examples/query5.png) |

Diccionario de columnas: [`documentation/DICCIONARIO_DATOS.md`](documentation/DICCIONARIO_DATOS.md).

In [11]:
from src.cassandra.demo import close_demo_session, display_cql_select, open_demo_session
from src.cassandra.selects import (
    critical_tickets_sla,
    daily_costs_and_requests,
    genai_tokens_daily,
    monthly_revenue,
    top_services_by_cost,
)
from src.cassandra.schema import TICKETS_CRITICAL_LOOKBACK_DAYS, TOP_SERVICES_LOOKBACK_DAYS

demo = open_demo_session(spark)

if demo is None:
    print("Astra not configured.")
else:
    p = demo.params
    print(f"DEMO_ORG_ID = {p.org_id}")
    print(f"#2 window = {p.period_start} → {p.period_end} ({TOP_SERVICES_LOOKBACK_DAYS}d)")
    print(f"#3 window = {p.q3_start} → {p.q3_end} ({TICKETS_CRITICAL_LOOKBACK_DAYS}d, severity={p.q3_severity})")


Astra not configured.


In [12]:
# Query #1 — daily costs and requests by service
if demo is None:
    print("Query #1 skipped.")
else:
    display_cql_select(
        demo.session,
        daily_costs_and_requests(demo.params.org_id, demo.params.q1_start, demo.params.q1_end),
    )



Query #1 skipped.


In [13]:
# Query #2 — top-N services by cost (rolling 14-day window)
if demo is None:
    print("Query #2 skipped.")
else:
    display_cql_select(
        demo.session,
        top_services_by_cost(
            demo.params.org_id, demo.params.period_end, demo.params.top_n,
        ),
    )



Query #2 skipped.


In [14]:
# Query #3 — high-severity tickets and SLA (severity=high)
if demo is None:
    print("Query #3 skipped.")
else:
    display_cql_select(
        demo.session,
        critical_tickets_sla(
            demo.params.org_id,
            demo.params.q3_severity,
            demo.params.q3_start,
            demo.params.q3_end,
        ),
    )



Query #3 skipped.


In [15]:
# Query #4 — monthly revenue USD
if demo is None:
    print("Query #4 skipped.")
else:
    display_cql_select(
        demo.session,
        monthly_revenue(demo.params.org_id, demo.params.q4_start, demo.params.q4_end),
    )



Query #4 skipped.


In [16]:
# Query #5 — GenAI tokens and estimated daily cost
if demo is None:
    print("Query #5 skipped.")
else:
    display_cql_select(
        demo.session,
        genai_tokens_daily(demo.params.org_id, demo.params.q5_start, demo.params.q5_end),
    )

close_demo_session(demo)



Query #5 skipped.
